# Read data from silver layer

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
from delta.tables import DeltaTable

In [0]:
df = spark.read.format("delta").load("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")
display(df)

# Dimension table for dealer info

In [0]:
df.createOrReplaceTempView("sales")

In [0]:
src_df = spark.sql("""
          select distinct Dealer_ID, DealerName
          from sales
          """)

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_dealer"):
    df_sink = spark.read.table("carsalescatalog.gold.dim_dealer")

else:
    df_sink = spark.sql("""
                         select 1 as dim_dealer_key, Dealer_ID, DealerName
                         from sales
                         where 1=0
                         """)


In [0]:
display(df_sink)

In [0]:
df_new_data = src_df.join(df_sink, src_df["Dealer_ID"] == df_sink["Dealer_ID"], "left")\
    .select(src_df["Dealer_ID"], src_df["DealerName"], df_sink["dim_dealer_key"])
df_new_data.display()

In [0]:
df_new_records = df_new_data.filter(df_new_data.dim_dealer_key.isNull())
df_old_records = df_new_data.filter(df_new_data.dim_dealer_key.isNotNull())
display(df_new_records)
display(df_old_records)

# Add surrogate dim key for new records

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_dealer"):
    max_value = spark.sql("""
                          select max(dim_dealer_key)
                          from carsalescatalog.gold.dim_dealer
                          """).collect()[0][0]

else:
    max_value = 0

In [0]:
df_new_records = df_new_records.withColumn("dim_dealer_key", max_value + monotonically_increasing_id() + 1)
display(df_new_records)

# Appending new records to old records

In [0]:
df_final = df_old_records.unionByName(df_new_records)
display(df_final)

# Insert or Update records (SCD Type 1)

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_dealer/"):
    deltatable = DeltaTable.forPath(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_dealer/")
    
    deltatable.alias("trg").merge(df_final.alias("src"), "trg.Dealer_ID = src.Dealer_ID")\
        .whenNotMatchedInsertAll()\
        .whenMatchedUpdateAll()\
        .execute()

else:
    df_final.write.format("delta").mode("overwrite").save("abfss://gold@adlscarsales.dfs.core.windows.net/dim_dealer/")

    spark.sql("""
              create table carsalescatalog.gold.dim_dealer
              using delta
              location 'abfss://gold@adlscarsales.dfs.core.windows.net/dim_dealer/'
              """)